# DS-07: Supplier/Plant Risk ML



## 📋 Contexto del Caso de Negocio

**Empresa:** "GlobalSupply Corp" - Empresa manufacturera con red global de proveedores y plantas de producción.

**Situación actual:**
- **Tasa de entregas tardías:** 12% (objetivo: <5%)
- **Problema:** Proveedores varían significativamente en calidad operacional (entregas tardías, defectos, variabilidad en lead times). El scoring manual de proveedores es subjetivo y reactivo, resultando en disrupciones costosas.
- Factores relevantes:
  - Evaluación de proveedores es subjetiva y manual
  - No hay sistema preventivo para identificar riesgos antes de disrupciones
  - Dependencia alta en 2-3 proveedores críticos sin alternativas validadas
  - Falta de métricas objetivas para auditorías y renegociaciones

**Impacto financiero:**
- Costo promedio de disrupciones: $50K-100K por evento
- Retrasos en línea de producción causan paradas y horas extras
- Defectos de calidad generan reproceso y garantías
- Pérdida de ventas por stockouts causados por proveedores no confiables

**Objetivo:** Implementar modelo de Machine Learning para clasificación automática de riesgo de proveedores para:
1. Predecir proveedores con alto riesgo operacional antes de hacer pedidos
2. Automatizar scoring mensual y generar alertas tempranas
3. Priorizar auditorías y acciones correctivas basadas en datos
4. Identificar proveedores alternativos confiables para diversificación

### 💼 ¿Por qué es IMPORTANTE?
- **Prevención vs Reacción:** Identificar riesgos ANTES de disrupciones costosas (paradas de línea, reproceso)
- **Objetividad:** Reemplazar scoring manual subjetivo con modelo basado en datos históricos
- **Eficiencia:** Priorizar recursos limitados de auditoría en proveedores de mayor riesgo
- **Continuidad:** Reducir dependencia en pocos proveedores identificando alternativas confiables

### 🎁 ¿PARA QUÉ sirve?
- **Sourcing:** Evaluar nuevos proveedores antes de contratos iniciales
- **Logistics:** Monitoreo continuo de desempeño y alertas automáticas de deterioro
- **Quality:** Identificar proveedores que requieren planes de mejora inmediatos
- **Strategic:** Diversificar base de proveedores reduciendo concentración de riesgo

### 🔧 ¿CÓMO se implementa?
- **Datos requeridos:** Historial de órdenes (dates, status), eventos de transporte (tracking), información de productos
- **Cálculo principal:** 
  - Features: `on_time_rate`, `delay_rate`, `cv_lead_time`, `total_orders`, `tracking_coverage`
  - Labels: `high_risk = 1` si `on_time_rate < 90%` O `delay_rate > 15%` O `cv_lead_time > 0.40`
- **Métrica resultado:** `ROC-AUC`, `Precision-Recall`, `Feature Importance`
- **Técnica aplicada:** Random Forest Classifier + Logistic Regression con hyperparameter tuning

---

## 🎯 Objetivos de Aprendizaje

- Extraer features predictivas desde datos transaccionales de órdenes y transporte
- Entrenar y comparar modelos de clasificación (Random Forest vs Logistic Regression)
- Evaluar performance con métricas apropiadas para clases desbalanceadas (ROC-AUC, Precision-Recall)
- Interpretar feature importance y aplicar el modelo a scoring de nuevos proveedores
- Implementar pipeline completo ML: feature engineering → modelado → evaluación → producción

## 📦 Instalación de Librerías Necesarias

**Antes de ejecutar este notebook, asegúrate de tener instaladas todas las dependencias.**

### Opción 1: Instalación dentro del notebook
Ejecuta la siguiente celda para instalar las librerías necesarias:

```python
%pip install pandas numpy scikit-learn plotly
```

### Opción 2: Instalación desde terminal
Si prefieres instalar desde la terminal, ejecuta:

```bash
# PowerShell o CMD
pip install pandas numpy scikit-learn plotly

# O si usas el proyecto completo con pyproject.toml
pip install -e .[core,notebooks]
```

### Librerías requeridas:
- `pandas`: Manipulación y análisis de datos
- `numpy`: Cálculos numéricos y arrays
- `scikit-learn`: Modelos de Machine Learning (Random Forest, Logistic Regression)
- `plotly`: Visualización interactiva de métricas y resultados

---

### 📝 Información del Notebook

| Campo | Valor |
| :--- | :--- |
| **🆔 ID** | `DS-07` |
| **📛 Título** | `Supplier/Plant Risk ML` |
| **🔹 Especialidad** | `Data Science & Machine Learning` |
| **⚙️ Proceso** | `Source (Supplier Management)` |
| **🧠 Nivel** | `Advanced` |
| **⏱️ Duración** | `45-60 min` |
| **🏷️ Etiquetas** | `machine-learning`, `classification`, `supplier-risk`, `random-forest`, `feature-engineering` |

---

## ⚙️ Configuración Inicial

In [16]:
# ⚙️ Configuración de rutas
import sys
from pathlib import Path

def resolve_repo_root():
    """Detecta raíz del repositorio buscando carpetas data/ y notebooks/"""
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / 'data').exists() and (parent / 'notebooks').exists():
            return parent
    return current

root = resolve_repo_root()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"✅ Rutas configuradas: {root}")

✅ Rutas configuradas: f:\GitHub\supply-chain-data-notebooks


In [17]:
# 📚 Importar librerías
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score
)

# Configuración de rutas de datos
DATA_DIR = root / "data" / "raw"
OUTPUT_DIR = root / "data" / "processed" / "ds07_supplier_risk_ml"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Librerías cargadas")
print(f"📁 Directorio datos: {DATA_DIR.resolve()}")
print(f"📂 Salida: {OUTPUT_DIR.resolve()}")

✅ Librerías cargadas
📁 Directorio datos: F:\GitHub\supply-chain-data-notebooks\data\raw
📂 Salida: F:\GitHub\supply-chain-data-notebooks\data\processed\ds07_supplier_risk_ml


---

# 🔧 PASOS DEL NOTEBOOK

---

## 📊 Paso 1: Cargar y Preparar Datos

**Técnica:** Ingesta de datasets raw y exploración inicial

**Datasets utilizados:**
- `locations.csv`: Ubicaciones geográficas (Plants, Stores, Warehouses)
- `orders.csv`: Historial de órdenes de compra
- `products.csv`: Catálogo de productos
- `transport_events.csv`: Eventos de seguimiento de transporte

In [18]:
# Cargar datasets
df_locations = pd.read_csv(DATA_DIR / "locations.csv")
df_orders = pd.read_csv(DATA_DIR / "orders.csv", parse_dates=['date'])
df_products = pd.read_csv(DATA_DIR / "products.csv")
df_transport = pd.read_csv(DATA_DIR / "transport_events.csv", parse_dates=['timestamp'])

print("📊 Datos Cargados:")
print(f"   - Locations: {len(df_locations):,} registros")
print(f"   - Orders: {len(df_orders):,} registros")
print(f"   - Products: {len(df_products):,} registros")
print(f"   - Transport Events: {len(df_transport):,} registros")

print(f"\n📍 Tipos de Ubicaciones:")
print(df_locations['type'].value_counts().to_string())

print(f"\n🚚 Estados de Transporte:")
print(df_transport['status'].value_counts().to_string())

print("\n✅ Paso 1 completado")

📊 Datos Cargados:
   - Locations: 30 registros
   - Orders: 8,504 registros
   - Products: 200 registros
   - Transport Events: 2,995 registros

📍 Tipos de Ubicaciones:
type
Store    21
DC        5
Hub       3
Plant     1

🚚 Estados de Transporte:
status
CREATED       1000
DISPATCHED    1000
IN_TRANSIT     670
DELIVERED      325

✅ Paso 1 completado


---

## 🔧 Paso 2: Feature Engineering

**Técnica:** Agregación de métricas por proveedor (Plant)

**Features calculadas:**
- `on_time_rate`: % de entregas a tiempo (objetivo: ≥90%)
- `delay_rate`: % de entregas con retraso
- `cv_lead_time`: Coeficiente de variación del lead time (variabilidad)
- `total_orders`: Volumen histórico de pedidos
- `tracking_coverage`: % de órdenes con eventos de seguimiento

**Criterio de riesgo:**
- `high_risk = 1` si `on_time_rate < 90%` O `delay_rate > 15%` O `cv_lead_time > 0.40`

In [19]:
def engineer_supplier_features(df_orders, df_transport, df_products):
    """
    Crear features predictivas para cada proveedor/plant basadas en DATOS REALES.
    
    Maneja el caso donde transport_events es parcial (no todos los órdenes tienen eventos).
    En producción: ~40% de órdenes tienen tracking completo.
    """
    
    # Fusionar órdenes con eventos de transporte (left join para mantener todas las órdenes)
    df_merged = df_orders.merge(df_transport, on='order_id', how='left')
    
    features_list = []
    
    for location_id in df_orders['location_id'].unique():
        orders_location = df_orders[df_orders['location_id'] == location_id].copy()
        
        if len(orders_location) == 0:
            continue
        
        merged_location = df_merged[df_merged['location_id'] == location_id].copy()
        
        # ===== FEATURES DE ÓRDENES (SIEMPRE DISPONIBLES) =====
        total_orders = len(orders_location)
        total_quantity = orders_location['qty'].sum()
        avg_order_quantity = orders_location['qty'].mean()
        std_order_quantity = orders_location['qty'].std() or 0
        
        unique_products = orders_location['sku'].nunique()
        unique_channels = orders_location['channel'].nunique()
        
        dominant_channel = orders_location['channel'].mode()[0] if len(orders_location['channel'].mode()) > 0 else 'Unknown'
        channel_concentration = orders_location['channel'].value_counts().iloc[0] / len(orders_location)
        
        days_span = (orders_location['date'].max() - orders_location['date'].min()).days
        days_since_last = (pd.Timestamp('2024-12-31') - orders_location['date'].max()).days
        
        # ===== FEATURES DE TRANSPORTE (PARCIALES - USAR SOLO CON ÓRDENES RASTREADAS) =====
        # Identificar órdenes con eventos de transporte
        orders_with_tracking = merged_location.dropna(subset=['status'])['order_id'].unique()
        tracked_ratio = len(orders_with_tracking) / total_orders if total_orders > 0 else 0
        
        if len(orders_with_tracking) > 0:
            # Usar SOLO órdenes rastreadas para calcular métricas de confiabilidad
            tracked_orders = merged_location[merged_location['order_id'].isin(orders_with_tracking)]
            
            # Obtener estado FINAL de cada orden (último estado)
            final_status = tracked_orders.sort_values('timestamp').drop_duplicates('order_id', keep='last')
            
            delivered = (final_status['status'] == 'DELIVERED').sum()
            in_transit = (final_status['status'] == 'IN_TRANSIT').sum()  # Trataremos como delay (no entregado a tiempo)
            created = (final_status['status'] == 'CREATED').sum()  # No enviado
            
            total_tracked = len(final_status)
            if total_tracked > 0:
                on_time_rate = delivered / total_tracked
                delay_rate = (in_transit + created) / total_tracked  # Órdenes que aún no se completan = delay
                failure_rate = 0.0  # No hay estado FAILED en datos
            else:
                on_time_rate = 0.5
                delay_rate = 0.5
                failure_rate = 0.0
            
            # Lead time real desde timestamps
            delivery_times = []
            for order_id in orders_with_tracking:
                order_events = merged_location[merged_location['order_id'] == order_id].sort_values('timestamp')
                if len(order_events) > 0:
                    created_ts = order_events[order_events['status'] == 'CREATED']['timestamp'].min()
                    final_ts = order_events['timestamp'].max()
                    if pd.notna(created_ts) and pd.notna(final_ts):
                        lead_time_hours = (final_ts - created_ts).total_seconds() / 3600
                        delivery_times.append(lead_time_hours)
            
            if len(delivery_times) > 0:
                avg_lead_time = np.mean(delivery_times)
                std_lead_time = np.std(delivery_times)
                cv_lead_time = std_lead_time / (avg_lead_time + 1e-6)
            else:
                avg_lead_time = 48
                std_lead_time = 0
                cv_lead_time = 0
        else:
            # Sin datos de transporte: usar estimaciones por defecto
            on_time_rate = 0.85  # Suponer 85% (conservative)
            delay_rate = 0.15
            failure_rate = 0.0
            avg_lead_time = 48
            std_lead_time = 12
            cv_lead_time = std_lead_time / avg_lead_time
            tracked_ratio = 0
        
        # ===== FEATURE DE CALIDAD =====
        # Combinar delay_rate con variabilidad de lead_time
        # Mayor variabilidad → menos predecible → riesgo de calidad
        defect_rate = (delay_rate * 0.5 + cv_lead_time * 0.1).clip(0, 0.25)
        quality_consistency = 1 - min(cv_lead_time, 1.0)  # 0-1: mayor = mejor
        
        # Calcular órdenes por mes
        months_active = max(days_span / 30, 1)
        orders_per_month = total_orders / months_active
        
        features = {
            'location_id': location_id,
            
            # VOLUMEN
            'total_orders': total_orders,
            'total_quantity': total_quantity,
            'avg_order_quantity': avg_order_quantity,
            'std_order_quantity': std_order_quantity,
            
            # DIVERSIDAD
            'unique_products': unique_products,
            'unique_channels': unique_channels,
            'dominant_channel': dominant_channel,
            'channel_concentration': channel_concentration,
            
            # CONFIABILIDAD (desde transport_events - parcial)
            'on_time_rate': on_time_rate,
            'delay_rate': delay_rate,
            'failure_rate': failure_rate,
            'tracking_coverage': tracked_ratio,  # % de órdenes con tracking
            
            # VARIABILIDAD
            'avg_lead_time_hours': avg_lead_time,
            'std_lead_time_hours': std_lead_time,
            'cv_lead_time': cv_lead_time,
            
            # CALIDAD Y CONSISTENCIA
            'defect_rate': defect_rate,
            'quality_consistency': quality_consistency,
            
            # RECENCY
            'days_since_last_order': days_since_last,
            'days_active': days_span,
            'orders_per_month': orders_per_month
        }
        
        features_list.append(features)
    
    df_features = pd.DataFrame(features_list)
    
    return df_features

# Generar features
df_features = engineer_supplier_features(df_orders, df_transport, df_products)

print(f"\n✅ Features generadas para {len(df_features)} locations")
print(f"\n📊 Dimensiones del dataset de features: {df_features.shape}")
print(f"\n🔍 Features disponibles ({len(df_features.columns)} total):")
for col in df_features.columns:
    print(f"   - {col}")

display(df_features.head(10))

# Estadísticas de features - con focus en métricas reales de riesgo
print("\n📈 Estadísticas de Confiabilidad (On-Time Rate):")
print(f"   Media: {df_features['on_time_rate'].mean():.2%}")
print(f"   Mín: {df_features['on_time_rate'].min():.2%}")
print(f"   Máx: {df_features['on_time_rate'].max():.2%}")
print(f"   Q25: {df_features['on_time_rate'].quantile(0.25):.2%}")
print(f"   Q75: {df_features['on_time_rate'].quantile(0.75):.2%}")

print("\n📈 Estadísticas de Delay Rate:")
print(f"   Media: {df_features['delay_rate'].mean():.2%}")
print(f"   Mín: {df_features['delay_rate'].min():.2%}")
print(f"   Máx: {df_features['delay_rate'].max():.2%}")

print("\n📈 Estadísticas de Lead Time Variability (CV):")
print(f"   Media: {df_features['cv_lead_time'].mean():.3f}")
print(f"   Mín: {df_features['cv_lead_time'].min():.3f}")
print(f"   Máx: {df_features['cv_lead_time'].max():.3f}")

print("\n📊 Cobertura de Tracking (% órdenes con eventos):")
print(f"   Media: {df_features['tracking_coverage'].mean():.1%}")
print(f"   Rango: {df_features['tracking_coverage'].min():.1%} - {df_features['tracking_coverage'].max():.1%}")


✅ Features generadas para 30 locations

📊 Dimensiones del dataset de features: (30, 21)

🔍 Features disponibles (21 total):
   - location_id
   - total_orders
   - total_quantity
   - avg_order_quantity
   - std_order_quantity
   - unique_products
   - unique_channels
   - dominant_channel
   - channel_concentration
   - on_time_rate
   - delay_rate
   - failure_rate
   - tracking_coverage
   - avg_lead_time_hours
   - std_lead_time_hours
   - cv_lead_time
   - defect_rate
   - quality_consistency
   - days_since_last_order
   - days_active
   - orders_per_month


,location_id,total_orders,total_quantity,avg_order_quantity,std_order_quantity,unique_products,unique_channels,dominant_channel,channel_concentration,on_time_rate,...,failure_rate,tracking_coverage,avg_lead_time_hours,std_lead_time_hours,cv_lead_time,defect_rate,quality_consistency,days_since_last_order,days_active,orders_per_month
0,LOC-013,311,2814,9.048232,7.339745,159,3,Retail,0.511254,0.212121,...,0.0,0.106109,10.727273,4.614028,0.430121,0.224830,0.569879,275,90,103.666667
1,LOC-011,269,2472,9.189591,6.348502,148,3,Retail,0.464684,0.342857,...,0.0,0.130112,12.000000,4.968472,0.414039,0.198547,0.585961,275,90,89.666667
2,LOC-019,263,2451,9.319392,6.417763,146,3,Retail,0.513308,0.390244,...,0.0,0.155894,12.439024,5.026994,0.404131,0.186755,0.595869,276,89,88.651685
3,LOC-023,308,3016,9.792208,7.310405,164,3,Retail,0.519481,0.321429,...,0.0,0.090909,12.000000,4.810702,0.400892,0.218661,0.599108,275,90,102.666667
4,LOC-025,277,2709,9.779783,7.088868,153,3,Retail,0.519856,0.418605,...,0.0,0.155235,12.697674,5.046458,0.397432,0.179278,0.602568,275,90,92.333333
5,LOC-020,260,2614,10.053846,7.230685,152,3,Retail,0.480769,0.451613,...,0.0,0.119231,13.161290,4.919328,0.373772,0.182539,0.626228,275,90,86.666667
6,LOC-027,296,2834,9.574324,6.874262,154,3,Retail,0.516892,0.312500,...,0.0,0.108108,12.000000,4.743416,0.395285,0.227028,0.604715,275,90,98.666667
7,LOC-001,285,2484,8.715789,7.091946,158,3,Retail,0.501754,0.433333,...,0.0,0.105263,13.600000,4.363485,0.320844,0.232084,0.679156,275,90,95.000000
8,LOC-026,291,2709,9.309278,7.020209,154,3,Retail,0.470790,0.312500,...,0.0,0.109966,11.812500,4.856938,0.411169,0.212992,0.588831,275,90,97.000000
9,LOC-029,314,2851,9.079618,6.651289,155,3,Retail,0.509554,0.303030,...,0.0,0.105096,11.272727,5.064868,0.449303,0.181294,0.550697,275,90,104.666667



📈 Estadísticas de Confiabilidad (On-Time Rate):
   Media: 32.70%
   Mín: 17.95%
   Máx: 46.43%
   Q25: 26.41%
   Q75: 37.00%

📈 Estadísticas de Delay Rate:
   Media: 34.39%
   Mín: 14.29%
   Máx: 47.06%

📈 Estadísticas de Lead Time Variability (CV):
   Media: 0.401
   Mín: 0.321
   Máx: 0.478

📊 Cobertura de Tracking (% órdenes con eventos):
   Media: 11.8%
   Rango: 8.8% - 15.6%


---

## 📊 Paso 3: Clasificación de Riesgo

**Técnica:** Definición de labels basada en umbrales de negocio

**Criterios de alto riesgo (ajustados por cobertura parcial de tracking):**
- `on_time_rate < 0.25`: Muy bajo desempeño en entregas rastreadas
- `delay_rate > 0.40`: Alto porcentaje de retrasos
- `cv_lead_time > 0.45`: Lead time muy variable
- `total_orders < 50`: Historial insuficiente

**Nota:** Los datos tienen ~12% de cobertura de tracking, en producción se busca >90%

In [20]:
# ===== DEFINIR RIESGO CON CRITERIOS DE NEGOCIO REALES =====
# Ajustado por cobertura parcial de tracking (solo ~12% de órdenes tienen eventos)

print("🎯 Definiendo criterios de riesgo...")
print("\n📋 Criterios de clasificación (ajustados por tracking parcial):\n")

criteria = {
    'on_time_rate < 0.25': '❌ Si entrega <25% a tiempo de rastreadas = CRÍTICO',
    'delay_rate > 0.40': '❌ Si >40% de rastreadas retrasadas = RIESGO ALTO',
    'cv_lead_time > 0.45': '❌ Si lead time muy variable (cv>0.45) = RIESGO',
    'total_orders < 50': '⚠️ Si <50 órdenes = historial insuficiente',
}

for criterion, desc in criteria.items():
    print(f"  {desc}")

print("\nℹ️ NOTA SOBRE LOS DATOS:")
print("  - Tracking coverage: ~11.8% de órdenes tienen eventos de transporte")
print("  - En producción: buscamos que >90% tengan tracking")
print("  - Los criterios se ajustan para reflejar datos parciales")

# Aplicar criterios RELAJADOS debido a cobertura de tracking limitada
df_features['is_high_risk'] = (
    (df_features['on_time_rate'] < 0.25) |      # Muy bajo on-time de las rastreadas
    (df_features['delay_rate'] > 0.40) |        # >40% retrasadas de las rastreadas
    (df_features['cv_lead_time'] > 0.45) |      # Lead time muy variable
    (df_features['total_orders'] < 50)          # Muy pocas órdenes
).astype(int)

# Distribución de target
risk_distribution = df_features['is_high_risk'].value_counts()
print("\n🎯 Distribución de Riesgo en Dataset:")
print(f"   - Bajo Riesgo (0): {risk_distribution.get(0, 0)} proveedores ({risk_distribution.get(0, 0) / len(df_features) * 100:.1f}%)")
print(f"   - Alto Riesgo (1): {risk_distribution.get(1, 0)} proveedores ({risk_distribution.get(1, 0) / len(df_features) * 100:.1f}%)")

# Mostrar ejemplos de cada categoría
low_risk_count = risk_distribution.get(0, 0)
high_risk_count = risk_distribution.get(1, 0)

if low_risk_count > 0:
    print("\n✅ EJEMPLOS DE BAJO RIESGO (proveedores confiables):")
    low_risk = df_features[df_features['is_high_risk'] == 0].nlargest(min(3, low_risk_count), 'on_time_rate')
    display(low_risk[['location_id', 'on_time_rate', 'delay_rate', 'cv_lead_time', 'total_orders', 'is_high_risk']])

if high_risk_count > 0:
    print("\n❌ EJEMPLOS DE ALTO RIESGO (proveedores problemáticos):")
    high_risk = df_features[df_features['is_high_risk'] == 1].nsmallest(min(3, high_risk_count), 'on_time_rate')
    display(high_risk[['location_id', 'on_time_rate', 'delay_rate', 'cv_lead_time', 'total_orders', 'is_high_risk']])

# Visualizar balance
if len(risk_distribution) == 2:
    fig = px.pie(
        values=risk_distribution.values,
        names=['Bajo Riesgo', 'Alto Riesgo'],
        title="Distribución de Riesgo de Proveedores (Basado en Criterios Ajustados)",
        color_discrete_sequence=['#2ecc71', '#e74c3c'],  # Verde-Rojo
        labels={'value': 'Cantidad'}
    )
    fig.update_traces(textposition='inside', textinfo='label+percent+value')
    fig.show()
else:
    print("\n⚠️ Distribución desbalanceada - usando barplot en su lugar")
    risk_counts = df_features['is_high_risk'].value_counts().sort_index()
    fig = px.bar(
        x=['Bajo Riesgo', 'Alto Riesgo'][:len(risk_counts)],
        y=risk_counts.values,
        title="Distribución de Riesgo de Proveedores",
        color=['#2ecc71', '#e74c3c'][:len(risk_counts)],
        labels={'x': 'Categoría de Riesgo', 'y': 'Cantidad'},
        text='y'
    )
    fig.update_traces(textposition='outside')
    fig.show()

print("\n✅ Paso 3 completado")

🎯 Definiendo criterios de riesgo...

📋 Criterios de clasificación (ajustados por tracking parcial):

  ❌ Si entrega <25% a tiempo de rastreadas = CRÍTICO
  ❌ Si >40% de rastreadas retrasadas = RIESGO ALTO
  ❌ Si lead time muy variable (cv>0.45) = RIESGO
  ⚠️ Si <50 órdenes = historial insuficiente

ℹ️ NOTA SOBRE LOS DATOS:
  - Tracking coverage: ~11.8% de órdenes tienen eventos de transporte
  - En producción: buscamos que >90% tengan tracking
  - Los criterios se ajustan para reflejar datos parciales

🎯 Distribución de Riesgo en Dataset:
   - Bajo Riesgo (0): 21 proveedores (70.0%)
   - Alto Riesgo (1): 9 proveedores (30.0%)

✅ EJEMPLOS DE BAJO RIESGO (proveedores confiables):


,location_id,on_time_rate,delay_rate,cv_lead_time,total_orders,is_high_risk
12,LOC-003,0.464286,0.142857,0.445615,253,0
5,LOC-020,0.451613,0.290323,0.373772,260,0
17,LOC-005,0.451613,0.354839,0.336852,254,0



❌ EJEMPLOS DE ALTO RIESGO (proveedores problemáticos):


,location_id,on_time_rate,delay_rate,cv_lead_time,total_orders,is_high_risk
19,LOC-018,0.179487,0.256410,0.477775,278,1
18,LOC-021,0.200000,0.400000,0.415740,272,1
0,LOC-013,0.212121,0.363636,0.430121,311,1



✅ Paso 3 completado


---

## 🔄 Paso 4: Preparar Datos para Modelado

**Técnica:** Train/test split con estratificación + normalización

**Parámetros:**
- Train/Test: 80/20
- Estratificación: Mantener proporción de clases
- Normalización: StandardScaler para Logistic Regression

**Features seleccionadas:** 16 features que capturan confiabilidad, variabilidad, capacidad y calidad

In [21]:
# ===== SELECCIONAR FEATURES PARA MODELADO =====
# Usar features que capturen RIESGO REAL (confiabilidad, variabilidad, calidad)

feature_cols = [
    # CONFIABILIDAD (más importante)
    'on_time_rate',
    'delay_rate',
    'failure_rate',
    
    # VARIABILIDAD
    'cv_lead_time',
    'avg_lead_time_hours',
    'std_lead_time_hours',
    
    # CAPACIDAD Y VOLUMEN
    'total_orders',
    'avg_order_quantity',
    'std_order_quantity',
    
    # DIVERSIDAD
    'unique_products',
    'unique_channels',
    'channel_concentration',
    
    # CALIDAD
    'defect_rate',
    'quality_consistency',
    
    # RECENCY
    'orders_per_month',
    'days_since_last_order'
]

print(f"📊 Features para modelado: {len(feature_cols)}")
for i, col in enumerate(feature_cols, 1):
    print(f"   {i:2d}. {col}")

X = df_features[feature_cols].copy()
y = df_features['is_high_risk'].copy()

# Manejar NaN (rellenar con 0 - indica "no data")
X = X.fillna(0)

# Split train/test (80/20) con estratificación (mantener proporciones)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 Datos preparados:")
print(f"   - Entrenamiento: {len(X_train)} proveedores")
print(f"   - Testing: {len(X_test)} proveedores")
print(f"   - Features: {len(feature_cols)}")

print(f"\n✅ Balance en training set (estratificado):")
print(f"   - Bajo Riesgo: {(y_train == 0).sum()} ({(y_train == 0).sum() / len(y_train) * 100:.1f}%)")
print(f"   - Alto Riesgo: {(y_train == 1).sum()} ({(y_train == 1).sum() / len(y_train) * 100:.1f}%)")

print(f"\n✅ Balance en testing set (esperado similar):")
print(f"   - Bajo Riesgo: {(y_test == 0).sum()} ({(y_test == 0).sum() / len(y_test) * 100:.1f}%)")
print(f"   - Alto Riesgo: {(y_test == 1).sum()} ({(y_test == 1).sum() / len(y_test) * 100:.1f}%)")

📊 Features para modelado: 16
    1. on_time_rate
    2. delay_rate
    3. failure_rate
    4. cv_lead_time
    5. avg_lead_time_hours
    6. std_lead_time_hours
    7. total_orders
    8. avg_order_quantity
    9. std_order_quantity
   10. unique_products
   11. unique_channels
   12. channel_concentration
   13. defect_rate
   14. quality_consistency
   15. orders_per_month
   16. days_since_last_order

📊 Datos preparados:
   - Entrenamiento: 24 proveedores
   - Testing: 6 proveedores
   - Features: 16

✅ Balance en training set (estratificado):
   - Bajo Riesgo: 17 (70.8%)
   - Alto Riesgo: 7 (29.2%)

✅ Balance en testing set (esperado similar):
   - Bajo Riesgo: 4 (66.7%)
   - Alto Riesgo: 2 (33.3%)


In [22]:
# Escalar features para Logistic Regression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Features escaladas")
print(f"   - Media (debe ser ~0): {X_train_scaled.mean():.6f}")
print(f"   - Desv.Est (debe ser ~1): {X_train_scaled.std():.6f}")

✅ Features escaladas
   - Media (debe ser ~0): -0.000000
   - Desv.Est (debe ser ~1): 0.935414


---

## 🌲 Paso 5: Entrenar Random Forest

**Técnica:** Random Forest Classifier (ensemble de 100 árboles)

**Hiperparámetros:**
- `n_estimators=100`: 100 árboles de decisión
- `max_depth=10`: Profundidad máxima (evita overfitting)
- `min_samples_split=5`: Mínimo 5 muestras para dividir nodo
- `min_samples_leaf=2`: Mínimo 2 muestras en hoja terminal

**Ventajas:** Robusto, maneja relaciones no-lineales, proporciona feature importance

In [23]:
# Random Forest Classifier
print("🌲 Entrenando Random Forest...")

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Predicciones
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Métricas
print("\n📊 Métricas - Random Forest:")
try:
    print(classification_report(y_test, y_pred_rf, target_names=['Bajo Riesgo', 'Alto Riesgo']))
except ValueError:
    # Si solo hay una clase, usar labels explícitamente
    print(classification_report(y_test, y_pred_rf, labels=[0, 1], target_names=['Bajo Riesgo', 'Alto Riesgo']))

try:
    roc_auc_rf = roc_auc_score(y_test, y_proba_rf)
    print(f"\n🎯 ROC-AUC Score: {roc_auc_rf:.3f}")
except ValueError as e:
    print(f"\n⚠️ ROC-AUC no disponible: {str(e)}")
    roc_auc_rf = None

# Matriz de confusión
cm_rf = confusion_matrix(y_test, y_pred_rf, labels=[0, 1])
fig = px.imshow(
    cm_rf,
    text_auto=True,
    labels=dict(x='Predicción', y='Actual', color='Casos'),
    x=['Bajo Riesgo', 'Alto Riesgo'],
    y=['Bajo Riesgo', 'Alto Riesgo'],
    title='Matriz de Confusión - Random Forest',
    color_continuous_scale='Blues'
)
fig.show()

🌲 Entrenando Random Forest...

📊 Métricas - Random Forest:
              precision    recall  f1-score   support

 Bajo Riesgo       1.00      0.50      0.67         4
 Alto Riesgo       0.50      1.00      0.67         2

    accuracy                           0.67         6
   macro avg       0.75      0.75      0.67         6
weighted avg       0.83      0.67      0.67         6


🎯 ROC-AUC Score: 1.000


---

## 📊 Paso 6: Entrenar Logistic Regression

**Técnica:** Logistic Regression con class_weight='balanced'

**Ventajas:**
- Modelo lineal interpretable con coeficientes claros
- Útil como baseline para comparación
- Maneja automáticamente desbalance de clases

**Ecuación:** $P(\text{Alto Riesgo}) = \frac{1}{1 + e^{-(w_1 \cdot x_1 + ... + w_n \cdot x_n + b)}}$

In [24]:
# Logistic Regression (con features escaladas)
print("📊 Entrenando Logistic Regression...")

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'  # Manejar desbalance
)

lr_model.fit(X_train_scaled, y_train)

# Predicciones
y_pred_lr = lr_model.predict(X_test_scaled)
y_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# Métricas
print("\n📊 Métricas - Logistic Regression:")
print(classification_report(y_test, y_pred_lr, target_names=['Bajo Riesgo', 'Alto Riesgo']))

roc_auc_lr = roc_auc_score(y_test, y_proba_lr)
print(f"\n🎯 ROC-AUC Score: {roc_auc_lr:.3f}")

# Matriz de confusión
cm_lr = confusion_matrix(y_test, y_pred_lr)
fig = px.imshow(
    cm_lr,
    text_auto=True,
    labels=dict(x="Predicción", y="Real", color="Count"),
    x=['Bajo Riesgo', 'Alto Riesgo'],
    y=['Bajo Riesgo', 'Alto Riesgo'],
    title="Matriz de Confusión - Logistic Regression",
    color_continuous_scale='Reds'
)
fig.show()

📊 Entrenando Logistic Regression...

📊 Métricas - Logistic Regression:
              precision    recall  f1-score   support

 Bajo Riesgo       1.00      0.75      0.86         4
 Alto Riesgo       0.67      1.00      0.80         2

    accuracy                           0.83         6
   macro avg       0.83      0.88      0.83         6
weighted avg       0.89      0.83      0.84         6


🎯 ROC-AUC Score: 1.000


---

## 📈 Paso 7: Curvas ROC y Precision-Recall

**Técnica:** Evaluación de rendimiento con métricas avanzadas

**Métricas:**
- **ROC-AUC**: Área bajo curva ROC (0.5=azar, 1.0=perfecto)
- **Precision-Recall**: Trade-off entre precisión y recall
- **Average Precision (AP)**: Resumen de Precision-Recall en un número

**Interpretación:** ROC-AUC mide probabilidad de clasificar correctamente un par aleatorio

### ⚠️ Limitaciones con Dataset Pequeño

Con solo **30 proveedores** (→ ~6 en test set), los resultados tienen **alta varianza**:
- ROC-AUC = 1.0 puede indicar **overfitting** (modelo memorizó, no aprendió)
- Cambiar el split train/test puede alterar resultados dramáticamente
- **Estos resultados son EDUCATIVOS**, no production-ready

**Para producción real se requiere:**
- Mínimo: 200-500 proveedores
- Recomendado: >1000 proveedores con validación cruzada

In [25]:
# ⚠️ NOTA IMPORTANTE SOBRE EL DATASET
print("⚠️  ADVERTENCIA: Dataset muy pequeño para ML robusto")
print(f"   - Total proveedores: {len(df_features)}")
print(f"   - Train set: {len(X_train)} proveedores")
print(f"   - Test set: {len(X_test)} proveedores")
print(f"\n   Para producción se recomienda: >1000 proveedores")
print(f"   Estos resultados son EDUCATIVOS, no production-ready\n")

# Curva ROC
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)

# Calcular ROC-AUC si hay suficientes datos
if len(np.unique(y_test)) < 2:
    print("⚠️ Test set tiene solo una clase - no se puede calcular ROC-AUC")
    roc_auc_rf = np.nan
    roc_auc_lr = np.nan
else:
    roc_auc_rf = roc_auc_score(y_test, y_proba_rf)
    roc_auc_lr = roc_auc_score(y_test, y_proba_lr)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fpr_rf, y=tpr_rf,
    mode='lines+markers',
    name=f'Random Forest (AUC={roc_auc_rf:.3f})',
    line=dict(color='blue', width=2),
    marker=dict(size=8)
))
fig.add_trace(go.Scatter(
    x=fpr_lr, y=tpr_lr,
    mode='lines+markers',
    name=f'Logistic Regression (AUC={roc_auc_lr:.3f})',
    line=dict(color='red', width=2),
    marker=dict(size=8)
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Random Baseline (AUC=0.5)',
    line=dict(color='gray', width=1, dash='dash')
))
fig.update_layout(
    title="Curva ROC - Comparación de Modelos<br><sub>⚠️ Dataset pequeño: resultados pueden variar significativamente</sub>",
    xaxis_title="False Positive Rate (FPR)",
    yaxis_title="True Positive Rate (TPR / Recall)",
    width=800, height=550,
    annotations=[
        dict(
            x=0.5, y=0.2,
            xref='x', yref='y',
            text=f'N_test={len(y_test)} (muy pequeño)',
            showarrow=False,
            font=dict(size=10, color='red'),
            opacity=0.6
        )
    ]
)
fig.show()

# Curva Precision-Recall
precision_rf, recall_rf, _ = precision_recall_curve(y_test, y_proba_rf)
precision_lr, recall_lr, _ = precision_recall_curve(y_test, y_proba_lr)
ap_rf = average_precision_score(y_test, y_proba_rf)
ap_lr = average_precision_score(y_test, y_proba_lr)

# Baseline para Precision-Recall (proporción de positivos)
baseline_precision = (y_test == 1).sum() / len(y_test)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=recall_rf, y=precision_rf,
    mode='lines+markers',
    name=f'Random Forest (AP={ap_rf:.3f})',
    line=dict(color='blue', width=2),
    marker=dict(size=8)
))
fig.add_trace(go.Scatter(
    x=recall_lr, y=precision_lr,
    mode='lines+markers',
    name=f'Logistic Regression (AP={ap_lr:.3f})',
    line=dict(color='red', width=2),
    marker=dict(size=8)
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[baseline_precision, baseline_precision],
    mode='lines',
    name=f'Random Baseline (AP={baseline_precision:.3f})',
    line=dict(color='gray', width=1, dash='dash')
))
fig.update_layout(
    title="Curva Precision-Recall<br><sub>⚠️ Dataset pequeño: resultados pueden variar significativamente</sub>",
    xaxis_title="Recall (True Positive Rate)",
    yaxis_title="Precision",
    width=800, height=550,
    annotations=[
        dict(
            x=0.5, y=baseline_precision + 0.1,
            xref='x', yref='y',
            text=f'N_test={len(y_test)} | Positivos={sum(y_test)}',
            showarrow=False,
            font=dict(size=10, color='red'),
            opacity=0.6
        )
    ]
)
fig.show()

print("\n📊 COMPARACIÓN DE MODELOS")
print("="*60)
print(f"Random Forest:")
print(f"  - ROC-AUC: {roc_auc_rf:.3f}")
print(f"  - Average Precision: {ap_rf:.3f}")
print(f"\nLogistic Regression:")
print(f"  - ROC-AUC: {roc_auc_lr:.3f}")
print(f"  - Average Precision: {ap_lr:.3f}")

print(f"\n⚠️  INTERPRETACIÓN CON DATASET PEQUEÑO (n={len(y_test)}):")
print("="*60)
if roc_auc_rf > 0.95:
    print("❌ ROC-AUC muy alto (>0.95) sugiere OVERFITTING")
    print("   → Modelo está memorizando, no generalizando")
    print("   → En producción con más datos, AUC bajará a 0.75-0.85")
elif roc_auc_rf > 0.85:
    print("✅ ROC-AUC bueno (0.85-0.95) pero verificar con más datos")
    print("   → Resultados preliminares prometedores")
else:
    print("🟡 ROC-AUC moderado (<0.85)")
    print("   → Puede mejorar con más datos o más features")

print(f"\n📈 Para resultados REALISTAS se necesita:")
print("   - Mínimo: 200-500 proveedores")
print("   - Recomendado: >1000 proveedores")
print("   - Validación cruzada (k-fold CV) con k=5 o k=10")
print("   - Test set estratificado con ~20-30% de datos")

print("\n✅ Paso 7 completado")

⚠️  ADVERTENCIA: Dataset muy pequeño para ML robusto
   - Total proveedores: 30
   - Train set: 24 proveedores
   - Test set: 6 proveedores

   Para producción se recomienda: >1000 proveedores
   Estos resultados son EDUCATIVOS, no production-ready




📊 COMPARACIÓN DE MODELOS
Random Forest:
  - ROC-AUC: 1.000
  - Average Precision: 1.000

Logistic Regression:
  - ROC-AUC: 1.000
  - Average Precision: 1.000

⚠️  INTERPRETACIÓN CON DATASET PEQUEÑO (n=6):
❌ ROC-AUC muy alto (>0.95) sugiere OVERFITTING
   → Modelo está memorizando, no generalizando
   → En producción con más datos, AUC bajará a 0.75-0.85

📈 Para resultados REALISTAS se necesita:
   - Mínimo: 200-500 proveedores
   - Recomendado: >1000 proveedores
   - Validación cruzada (k-fold CV) con k=5 o k=10
   - Test set estratificado con ~20-30% de datos

✅ Paso 7 completado


---

## 🔍 Paso 8: Feature Importance

**Técnica:** Análisis de importancia de features y coeficientes

**Objetivo:** Identificar qué variables son más predictivas del riesgo

**Interpretación:**
- Random Forest: Importancia basada en reducción de impureza
- Logistic Regression: Coeficientes (positivo=aumenta riesgo, negativo=reduce riesgo)

In [26]:
# Feature importance de Random Forest
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("🔍 Feature Importance (Random Forest):")
display(feature_importance)

# Visualizar
fig = px.bar(
    feature_importance,
    x='importance',
    y='feature',
    orientation='h',
    title="Feature Importance - Random Forest",
    labels={'importance': 'Importancia', 'feature': 'Feature'},
    color='importance',
    color_continuous_scale='Viridis'
)
fig.update_layout(height=500, showlegend=False)
fig.show()

# Coeficientes de Logistic Regression
lr_coefficients = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': lr_model.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False)

print("\n📊 Coeficientes (Logistic Regression):")
display(lr_coefficients)

fig = px.bar(
    lr_coefficients,
    x='coefficient',
    y='feature',
    orientation='h',
    title="Coeficientes - Logistic Regression",
    labels={'coefficient': 'Coeficiente', 'feature': 'Feature'},
    color='coefficient',
    color_continuous_scale='RdBu_r'
)
fig.update_layout(height=500)
fig.show()

🔍 Feature Importance (Random Forest):


,feature,importance
5,std_lead_time_hours,0.325885
12,defect_rate,0.158821
4,avg_lead_time_hours,0.138756
1,delay_rate,0.104485
0,on_time_rate,0.095720
7,avg_order_quantity,0.051729
11,channel_concentration,0.051631
9,unique_products,0.019634
13,quality_consistency,0.016669
3,cv_lead_time,0.013202



📊 Coeficientes (Logistic Regression):


,feature,coefficient
5,std_lead_time_hours,-1.163094
0,on_time_rate,-0.844079
11,channel_concentration,0.781697
4,avg_lead_time_hours,-0.703614
7,avg_order_quantity,-0.373226
1,delay_rate,0.359867
9,unique_products,-0.335335
6,total_orders,-0.279257
12,defect_rate,0.248248
14,orders_per_month,-0.235637


---

## 🔧 Paso 9: Hyperparameter Tuning (Opcional)

**Técnica:** GridSearchCV con validación cruzada

**Hiperparámetros probados:**
- `n_estimators`: [50, 100, 200]
- `max_depth`: [5, 10, 15]
- `min_samples_split`: [2, 5, 10]

**Total:** 27 combinaciones × 3-fold CV = 81 entrenamientos

In [27]:
# GridSearchCV para Random Forest (puede tardar varios minutos)
print("🔧 Hyperparameter Tuning con GridSearchCV...")
print("   (Esto puede tardar 1-2 minutos)\n")

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=3,
    scoring='roc_auc',
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f"\n✅ Mejores hiperparámetros:")
print(grid_search.best_params_)
print(f"\n🎯 Mejor ROC-AUC (CV): {grid_search.best_score_:.3f}")

# Evaluar modelo optimizado
best_rf = grid_search.best_estimator_
y_proba_best = best_rf.predict_proba(X_test)[:, 1]
roc_auc_best = roc_auc_score(y_test, y_proba_best)

print(f"🎯 ROC-AUC en Test (modelo optimizado): {roc_auc_best:.3f}")

🔧 Hyperparameter Tuning con GridSearchCV...
   (Esto puede tardar 1-2 minutos)

Fitting 3 folds for each of 27 candidates, totalling 81 fits

✅ Mejores hiperparámetros:
{'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 100}

🎯 Mejor ROC-AUC (CV): 1.000
🎯 ROC-AUC en Test (modelo optimizado): 1.000


---

## 🎯 Paso 10: Generar Risk Scoring

**Técnica:** Aplicar modelo a todos los proveedores

**Risk Score:** Probabilidad de alto riesgo × 100 (0-100)

**Interpretación:**
- 0-20: 🟢 Bajo Riesgo (confiable)
- 20-50: 🟡 Riesgo Medio (monitorear)
- 50-100: 🔴 Alto Riesgo (acciones inmediatas)

In [28]:
# Generar scoring para TODOS los locations
X_all = df_features[feature_cols].fillna(0)
risk_scores = rf_model.predict_proba(X_all)[:, 1]
risk_predictions = rf_model.predict(X_all)

# Agregar a dataframe
df_features['risk_score'] = risk_scores
df_features['predicted_risk'] = risk_predictions
df_features['risk_category'] = pd.cut(
    risk_scores,
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Bajo', 'Medio', 'Alto']
)

# Top 10 locations de mayor riesgo
print("🚨 Top 10 Locations de MAYOR RIESGO:")
top_risk = df_features.nlargest(10, 'risk_score')[[
    'location_id', 'risk_score', 'risk_category', 'delay_rate', 'defect_rate', 'cv_lead_time'
]]
display(top_risk)

# Distribución de scoring
fig = px.histogram(
    df_features,
    x='risk_score',
    color='risk_category',
    title="Distribución de Risk Score",
    labels={'risk_score': 'Risk Score', 'count': 'Frecuencia'},
    color_discrete_map={'Bajo': 'green', 'Medio': 'yellow', 'Alto': 'red'}
)
fig.show()

🚨 Top 10 Locations de MAYOR RIESGO:


,location_id,risk_score,risk_category,delay_rate,defect_rate,cv_lead_time
18,LOC-021,0.939000,Alto,0.400000,0.241574,0.415740
21,LOC-012,0.853714,Alto,0.419355,0.248906,0.392287
0,LOC-013,0.849405,Alto,0.363636,0.224830,0.430121
28,LOC-009,0.825714,Alto,0.459459,0.250000,0.331662
23,LOC-024,0.824000,Alto,0.428571,0.250000,0.393700
15,LOC-010,0.817381,Alto,0.470588,0.250000,0.339071
10,LOC-017,0.756548,Alto,0.435897,0.250000,0.375534
19,LOC-018,0.735000,Alto,0.256410,0.175983,0.477775
20,LOC-002,0.698333,Alto,0.363636,0.224188,0.423700
7,LOC-001,0.686000,Alto,0.400000,0.232084,0.320844


In [29]:
# Exportar resultados
import pickle

# Guardar features con scores
df_features['risk_score_rf'] = rf_model.predict_proba(df_features[feature_cols].fillna(0))[:, 1] * 100
df_features.to_csv(OUTPUT_DIR / 'supplier_risk_scores.csv', index=False)

# Guardar modelo
model_package = {
    'model': rf_model,
    'feature_cols': feature_cols,
    'model_type': 'RandomForestClassifier'
}

with open(OUTPUT_DIR / 'supplier_risk_model.pkl', 'wb') as f:
    pickle.dump(model_package, f)

print(f"✅ Resultados exportados a {OUTPUT_DIR}")
print(f"   - supplier_risk_scores.csv: Features y scores")
print(f"   - supplier_risk_model.pkl: Modelo entrenado")

✅ Resultados exportados a f:\GitHub\supply-chain-data-notebooks\data\processed\ds07_supplier_risk_ml
   - supplier_risk_scores.csv: Features y scores
   - supplier_risk_model.pkl: Modelo entrenado


In [30]:
# Validaciones de integridad
assert len(df_features) > 0, "No se generaron features"
assert 'is_high_risk' in df_features.columns, "Falta columna de target"
assert 'risk_score_rf' in df_features.columns, "Falta columna de risk score"
assert df_features['risk_score_rf'].between(0, 100).all(), "Risk scores fuera de rango"
assert (OUTPUT_DIR / 'supplier_risk_model.pkl').exists(), "Modelo no guardado"

print("✅ Validaciones pasadas")
print("✅ Notebook DS-07 completado: Modelo de riesgo de proveedores entrenado y exportado")

✅ Validaciones pasadas
✅ Notebook DS-07 completado: Modelo de riesgo de proveedores entrenado y exportado


---

## 📚 Resumen Técnico y Referencias



### 🎯 Resultados Clave

Este análisis implementa un modelo de Machine Learning para clasificación de riesgo de proveedores basado en métricas operacionales.

**Componentes calculados:**
1. **Features de Confiabilidad**: `on_time_rate`, `delay_rate`, `failure_rate` - capturan desempeño de entrega
2. **Features de Variabilidad**: `cv_lead_time`, `avg_lead_time_hours`, `std_lead_time_hours` - miden consistencia
3. **Features de Capacidad**: `total_orders`, `avg_order_quantity` - indican experiencia y volumen
4. **Target Binario**: `is_high_risk` (0=Bajo, 1=Alto) basado en umbrales de negocio

**Hallazgos típicos:**
- On-time rate promedio: ~32.7% (datos parciales con 11.8% tracking coverage)
- Lead time variability (CV) promedio: 0.40
- Distribución: ~70% Bajo Riesgo, ~30% Alto Riesgo

**Clasificación:**
- **Bajo Riesgo (0)**: `on_time_rate ≥ 25%` Y `delay_rate ≤ 40%` Y `cv_lead_time ≤ 0.45` → Proveedor confiable
- **Alto Riesgo (1)**: Cualquier criterio violado → Requiere auditoría y acciones correctivas

### 🔬 Metodología

**Modelos entrenados:**
1. **Random Forest Classifier** (ensemble de 100 árboles)
   - Hiperparámetros: `n_estimators=100`, `max_depth=10`, `min_samples_split=5`
   - ROC-AUC: ~1.0 (excelente discriminación)
   - Feature importance disponible para interpretación

2. **Logistic Regression** (modelo lineal)
   - Con normalización (StandardScaler)
   - `class_weight='balanced'` para manejo de desbalance
   - Coeficientes interpretables por stakeholders

**Evaluación:**
- Train/Test Split: 80/20 con estratificación
- Métricas: ROC-AUC, Precision-Recall, Confusion Matrix
- Hyperparameter tuning con GridSearchCV (opcional)

### 📖 Aplicaciones Prácticas

1. **Evaluación de nuevos proveedores:**
   - Calcular features desde historial de pedidos de prueba
   - Predecir risk score antes de contrato largo plazo

2. **Monitoreo continuo:**
   - Actualización mensual/semanal de features
   - Alertas automáticas cuando risk score aumenta >10%

3. **Priorización de auditorías:**
   - Ranking de proveedores por risk score
   - Asignar recursos a los 5-10 proveedores de mayor riesgo

### 🔗 Referencias

1. **Chopra, S., & Meindl, P. (2016)**. *Supply Chain Management: Strategy, Planning, and Operation*. Pearson.
   - Fundamentos de evaluación de proveedores y gestión de riesgo

2. **Hastie, T., Tibshirani, R., & Friedman, J. (2009)**. *The Elements of Statistical Learning*. Springer.
   - Random Forest y técnicas de ensemble para clasificación

3. **He, H., & Garcia, E. A. (2009)**. *Learning from Imbalanced Data*. IEEE Transactions on Knowledge and Data Engineering.
   - Técnicas para manejo de clases desbalanceadas

### 💡 Extensiones Futuras

- Incorporar temporal trends (¿está mejorando o empeorando?)
- Agregar datos externos (financial health, geopolitical risk)
- Implementar SHAP values para explicabilidad individual
- Model monitoring y re-entrenamiento automático
- Modelos específicos por categoría de producto

---

**Autor**: lraigosov (@LuisRai)  
**Fecha**: 2025  
**Versión**: 1.0  
**Tags**: `#machine-learning` `#classification` `#supplier-risk` `#random-forest` `#supply-chain`

---

<div style="width: 100%; clear: both; margin: 0 0 20px 0; border-top: 1px solid #eaecef; padding-top: 24px;"><div style="display: flex; justify-content: space-between; align-items: center; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif;"><div style="flex: 1; text-align: left;"><a href="DS-06-forecast_arima.ipynb" style="text-decoration: none; color: #0366d6; font-size: 14px; font-weight: 600; transition: color 0.2s;">← Anterior: [DS-06-forecast_arima.ipynb](../30_data_science_ml/DS-06-forecast_arima.ipynb)</a></div><div style="flex: 1; text-align: center; font-size: 14px;"><a href="../../README.md" style="color: #0366d6; text-decoration: none; font-weight: 600; margin: 0 10px;">📑 Índice</a><span style="color: #6a737d;">|</span><a href="../../config/notebooks_index.yml" style="color: #0366d6; text-decoration: none; font-weight: 600; margin: 0 10px;">📋 Catálogo</a></div><div style="flex: 1; text-align: right;"><span style="color: #6a737d; font-size: 14px; cursor: default;">Siguiente →</span></div></div></div>

